In [1]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# 1. Визуализация и предобработка

Прогнозируем максимальную температуру следующего дня по предыдущим 30 дням.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CSV_PATH = DATA_DIR / "seattle-weather.csv"
SOURCE_URL = "https://www.kaggle.com/datasets/ananthr1/weather-prediction"

NUMERIC_COLUMNS = ["precipitation", "temp_max", "temp_min", "wind"]
TARGET_COLUMN = "temp_max"
LOOKBACK = 30
TRAIN_END = pd.Timestamp("2014-01-01")
VALIDATION_END = pd.Timestamp("2015-01-01")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 16,
    "font.family": "Times New Roman",
})

Чтение локального CSV

In [9]:
df = pd.read_csv(CSV_PATH, parse_dates=["date"])
raw_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()

print(f"Источник: {SOURCE_URL}")
print(f"Размер таблицы: {df.shape[0]} строк, {df.shape[1]} столбцов")
display(df.head())

Источник: https://www.kaggle.com/datasets/ananthr1/weather-prediction
Размер таблицы: 1461 строк, 6 столбцов


,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


## 2. Проверка качества данных


In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=["date"])

expected = {'date', 'precipitation', 'temp_max', 'temp_min', 'wind', 'weather'}

df = df.sort_values('date').set_index('date')
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in NUMERIC_COLUMNS)
calendar = pd.date_range(df.index.min(), df.index.max(), freq='D')
missing_dates = calendar.difference(df.index)

print('Пропущенных дат:', len(missing_dates), 'Дубликатов строк:', df.reset_index().duplicated().sum())

display(df.isna().sum().rename('Пропуски'))
display(df[NUMERIC_COLUMNS].describe().round(2))

assert len(missing_dates) == 0, 'Необходимо отдельно определить стратегию заполнения дат'
assert np.isfinite(df[NUMERIC_COLUMNS].to_numpy()).all(), 'Пропуски/inf требуют отдельной обработки'
assert (df['temp_min'] <= df['temp_max']).all()
assert (df[['precipitation', 'wind']] >= 0).all().all()


Пропущенных дат: 0 Дубликатов строк: 0


precipitation    0
temp_max         0
temp_min         0
wind             0
weather          0
Name: Пропуски, dtype: int64

,precipitation,temp_max,temp_min,wind
count,1461.00,1461.00,1461.00,1461.00
mean,3.03,16.44,8.23,3.24
std,6.68,7.35,5.02,1.44
min,0.00,-1.60,-7.10,0.40
25%,0.00,10.60,4.40,2.20
50%,0.00,15.60,8.30,3.00
75%,2.80,22.20,12.20,4.00
max,55.90,35.60,18.30,9.50


## Разделение по времени — до выбора модели

Обучение: **2012–2013**, валидация: **2014**, тест: **2015**.
Каждый период валидации/теста покрывает полный сезонный цикл.
На train вычисляем параметры преобразований и обучаем веса; по validation выбираем
эпоху; test открываем для итогового сравнения в третьем ноутбуке.
Случайный `train_test_split` здесь не подходит: он смешивает прошлое с будущим.

Проверка схемы/пропусков выше охватывает весь файл, а графики и анализ закономерностей
ниже — только train. Это ограничивает подглядывание в тест при разработке.


In [ ]:
train_mask = df.index < TRAIN_END
validation_mask = (df.index >= TRAIN_END) & (df.index < VALIDATION_END)
test_mask = df.index >= VALIDATION_END

train = df.loc[train_mask].copy()
split_summary = pd.DataFrame({
    "Период": ["train", "validation", "test"],
    "Количество дней": [train_mask.sum(), validation_mask.sum(), test_mask.sum()],
})
display(split_summary)


## 1.4. Изменения погоды во времени


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

train[["temp_max", "temp_min"]].plot(ax=axes[0], alpha=0.6)
temperature_mean_30 = train[TARGET_COLUMN].rolling(window=30).mean()
temperature_mean_30.plot(ax=axes[0], color="black", label="Среднее за прошлые 30 дней")
axes[0].set_ylabel("Температура, °C")
axes[0].legend()

train["precipitation"].plot(ax=axes[1])
axes[1].set_ylabel("Осадки, мм")

train["wind"].plot(ax=axes[2])
axes[2].set_ylabel("Ветер, м/с")

fig.suptitle("Только обучающий период")
plt.tight_layout()
plt.show()


## 1.5. Сезонность, распределения и зависимости


In [ ]:
lags = range(1, 61)
autocorrelations = [train[TARGET_COLUMN].autocorr(lag) for lag in lags]
train_by_month = train.assign(month=train.index.month)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

train_by_month.boxplot(column="temp_max", by="month", ax=axes[0])
axes[0].set(title="Температура по месяцам", xlabel="Месяц", ylabel="°C")

train["precipitation"].hist(bins=35, ax=axes[1])
axes[1].set(title="Распределение осадков", xlabel="мм", ylabel="Количество дней")

axes[2].plot(list(lags), autocorrelations)
axes[2].set(title="Автокорреляция температуры", xlabel="Лаг, дни", ylabel="Корреляция")

fig.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
display(train[NUMERIC_COLUMNS].corr().round(2))
display(pd.crosstab(train.index.year, train["weather"]))

dry_days_fraction = (train["precipitation"] == 0).mean()
lag_one_correlation = train[TARGET_COLUMN].autocorr(lag=1)

print(f"Доля дней без осадков на train: {dry_days_fraction:.1%}")
print(f"Корреляция температуры с предыдущим днём: {lag_one_correlation:.3f}")


## Выводы и решения по предобработке

* В исходном файле 1461 дата от 2012-01-01 до 2015-12-31, без пропусков.
  Заполнять или удалять наблюдения в этой версии не требуется.
* На обучающем графике заметен годовой цикл температуры. Добавим синус и косинус
  положения внутри календарного года: декабрь и январь будут соседними по признакам.
* Соседние температуры связаны; это обосновывает окно истории и сильный baseline
  «как вчера». Автокорреляция частично объясняется сезонностью и не доказывает причинность.
* Осадки асимметричны, с большим числом нулей. Используем `log1p`, затем стандартизацию.
  Большие осадки сами по себе не ошибка: экстремумы не удаляем автоматически.
* Таблица категорий по годам позволяет увидеть нестабильность частот `weather`.
  Причину по этому CSV установить нельзя; категорию в первую модель не включаем.
* Четыре года — мало для глубокой сети и вывода о климатическом тренде.
  Ограничим размер LSTM и применим раннюю остановку.

Среднее и стандартное отклонение считаем **только по train**. Если в новой версии
будут пропуски, сначала исследуйте их причину: двусторонняя интерполяция способна
использовать будущее. Не заполняйте пропущенные целевые значения для оценки качества.


## 1.6. Создание признаков


In [ ]:
features = df[NUMERIC_COLUMNS].astype(float).copy()
features["precipitation"] = np.log1p(features["precipitation"])

# Декабрь и январь должны оставаться соседями на годовом цикле.
days_in_year = np.where(features.index.is_leap_year, 366, 365)
year_phase = 2 * np.pi * (features.index.dayofyear - 1) / days_in_year
features["year_sin"] = np.sin(year_phase)
features["year_cos"] = np.cos(year_phase)

display(features.head())


## 1.7. Масштабирование только по обучающему периоду


In [ ]:
training_features = features.loc[train_mask]
x_mean = training_features.mean()
x_std = training_features.std(ddof=0).replace(0, 1)

y_mean = float(train[TARGET_COLUMN].mean())
y_std = float(train[TARGET_COLUMN].std(ddof=0))

scaled_x = ((features - x_mean) / x_std).to_numpy(dtype=np.float32)
scaled_y = ((df[TARGET_COLUMN] - y_mean) / y_std).to_numpy(dtype=np.float32)


## 1.8. Временные окна: 30 прошлых дней → следующий день


In [ ]:
input_windows = []
target_values = []

for target_position in range(LOOKBACK, len(df)):
    window_start = target_position - LOOKBACK
    # Правая граница среза не включается: день цели отсутствует во входе.
    input_windows.append(scaled_x[window_start:target_position])
    target_values.append(scaled_y[target_position])

X = np.stack(input_windows)
y = np.asarray(target_values, dtype=np.float32)
target_dates = df.index[LOOKBACK:]

print(f"Вход X: {X.shape} — примеры, дни, признаки")
print(f"Цель y: {y.shape} — одна температура на пример")


## 1.9. Разделение окон по дате прогнозируемого дня


In [ ]:
split_masks = {
    "train": target_dates < TRAIN_END,
    "val": (target_dates >= TRAIN_END) & (target_dates < VALIDATION_END),
    "test": target_dates >= VALIDATION_END,
}

arrays = {}
for split_name, split_mask in split_masks.items():
    split_dates = target_dates[split_mask]
    arrays[f"X_{split_name}"] = X[split_mask]
    arrays[f"y_{split_name}"] = y[split_mask]
    arrays[f"dates_{split_name}"] = split_dates.strftime("%Y-%m-%d").to_numpy(dtype="U10")

    print(
        f"{split_name}: {split_mask.sum()} примеров, "
        f"с {split_dates.min().date()} по {split_dates.max().date()}"
    )

assert X.shape[1:] == (LOOKBACK, len(features.columns))
assert np.isfinite(X).all() and np.isfinite(y).all()
assert target_dates[split_masks["train"]].max() < target_dates[split_masks["val"]].min()
assert target_dates[split_masks["val"]].max() < target_dates[split_masks["test"]].min()


## 1.10. Сохранение данных для следующих ноутбуков


In [ ]:
preprocessing = {
    "target": TARGET_COLUMN,
    "lookback": LOOKBACK,
    "horizon": 1,
    "features": list(features.columns),
    "x_mean": x_mean.to_dict(),
    "x_std": x_std.to_dict(),
    "y_mean": y_mean,
    "y_std": y_std,
    "train_end_exclusive": str(TRAIN_END.date()),
    "val_end_exclusive": str(VALIDATION_END.date()),
    "source": SOURCE_URL,
    "raw_sha256": raw_sha256,
}

np.savez_compressed(ARTIFACTS_DIR / "prepared.npz", **arrays)
df.to_csv(ARTIFACTS_DIR / "observations.csv")

metadata_path = ARTIFACTS_DIR / "preprocessing.json"
metadata_path.write_text(
    json.dumps(preprocessing, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Результаты сохранены в {ARTIFACTS_DIR}")


Форма входа: **(примеры, 30 дней, 6 признаков)**; выход: одна температура на пример.
Первые 30 дней используются как история, поэтому train содержит 701 пример.
История для первых дней validation/test может включать предыдущий период: это нормально,
поскольку она уже известна. Деление проводится по **дате цели**.

Оценка будет в режиме ежедневного обновления: перед каждым прогнозом доступны все
фактические измерения предыдущего дня, включая прошлые дни тестового года.
Это не прогноз всего 2015 года из одной точки 31 декабря 2014 года.

Источник: [Kaggle — Weather Prediction](https://www.kaggle.com/datasets/ananthr1/weather-prediction).
Следующий шаг — ноутбук 2.
